# Phase 4 Probe (launcher)

## 1 — Secrets, clone

In [1]:
import getpass, os
github_token = getpass.getpass('GitHub PAT (repo scope): ')
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [2]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    print('check')
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

check
Cloning into '/content/Longor'...
remote: Enumerating objects: 846, done.
remote: Counting objects: 100% (597/597), done.
remote: Compressing objects: 100% (292/292), done.
remote: Total 846 (delta 365), reused 459 (delta 294), pack-reused 249 (from 1)
Receiving objects: 100% (846/846), 513.77 KiB | 8.29 MiB/s, done.
Resolving deltas: 100% (485/485), done.


## 2 — Headless rendering + app deps

In [3]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-libero.txt
%pip install -e .

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [355 B]       
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,850 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]          
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy I

Obtaining file:///content/Longor
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for longor (pyproject.toml) ... done
  Created wheel for longor: filename=longor-0.0.0-0.editable-py3-none-any.whl size=2652 sha256=d5f913da47cbdb874d0e7bcd7690205e0aeb33439b342e59b3dd32584b1b2dfc
  Stored in directory: /tmp/pip-ephem-wheel-cache-nng2s1u9/wheels/9c/b6/02/742808f85af524454576afd175a8131f318804aef665dd4760
Successfully built longor


## 3 — SmolVLA venv

In [4]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'

def vla_pip(*args):
    subprocess.run([VLA_PYTHON, '-m', 'pip', *args], check=True)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
vla_pip('install', '-U', 'pip')
vla_pip('install', '-r', 'deps/requirements-vla-smolvla-linux.txt')
vla_pip('install', '-e', '.')

## 4 — `.env` + start VLA server + health

In [11]:
import os, subprocess, time, urllib.request

with open('.env', 'w', encoding='utf-8') as f:
    f.write('VLA_POLICY=smolvla\nMUJOCO_GL=egl\nLOG_LEVEL=DEBUG\n')

VLA_LOG = '/tmp/m7_wiring_vla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
log = open(VLA_LOG, 'w', buffering=1)
proc = subprocess.Popen(
    [vla_python, 'services/smolvla_service.py'],
    cwd='/content/Longor',
    stdout=log,
    stderr=subprocess.STDOUT,
    env={**os.environ, 'VLA_POLICY': 'smolvla'},
)

health_ok = False
for elapsed in range(0, 180, 5):
    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('Health:', resp.read().decode())
        health_ok = True
        break
    except Exception:
        print(f'Waiting for /health ({elapsed}s)...')
        time.sleep(5)

if not health_ok:
    raise RuntimeError(f'VLA server not healthy; see {VLA_LOG}')

Waiting for /health (0s)...
Waiting for /health (5s)...
Waiting for /health (10s)...
Waiting for /health (15s)...
Waiting for /health (20s)...
Waiting for /health (25s)...
Waiting for /health (30s)...
Health: {"status":"ok","model":"smolvla"}


## 5 — Run probe

In [12]:
!echo -e "n\n" | python examples/long_horizon_llm_libero.py

[robosuite WARNING] No private macro file found! (__init__.py:7)
WARNING [robosuite_logs] No private macro file found!
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
WARNING [robosuite_logs] It is recommended to use a private macro file
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
WARNING [robosuite_logs] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Local assets not found. Downloading from HuggingFace Hub...
Assets already downloaded at /root/.cache/libero/assets
INFO [libero_env] LIBERO ready: libero_10 task 0 - 'put both the alphabet soup and the tomato sauce in the basket'
INFO [runner] Scene objects: ['alphabet_soup_1', 'tomato_sauce_1', 'basket_1']
DEBUG [planner] LLMPlanner: attempt 1/3 for goal='put the alphabet soup can and the cream cheese box in the b